In [ ]:
#this may not be necessary
rm(list = ls(all = TRUE)) 
#install.packages("ggpubr")
library(ggplot2)
library(behavr)
library(scopr)
library(sleepr)
library(ggetho)
library(plotly)
#library(survival)
library(cowplot)
#library(ggthemes)
library(plotly)
library(data.table)
library(stringi)
library(ggtern)
library(ggpubr)
library(EnvStats)
library(RColorBrewer)
library(dplyr)
library(plyr)
library(Hmisc)

In [ ]:
REMOTE_DATA_SOURCE <- "ftp://turing.lab.gilest.ro/auto_generated_data/ethoscope_results/"
MY_DIR <- "/home/hjones/insecticide_movement/Qs_data/"
setwd(MY_DIR)

# This is the placement of the data in this computer, make sure this is a file where you want it saved!
DATA_DIR <- "/mnt/ethoscope_results"
#This is the placement of the real data, on the NAS

#this is the place were a cache version of the data is stored, once it has been taken from NAS. This make the loading faster on the second time.
CACHE <- "/home/cache"

#This is the query of the experiment (this is the table you made of the data)
METADATA <- "/home/hjones/Qs_data_okay_only_F.csv"

In [ ]:
#To get the files from the remote source
query <- link_ethoscope_metadata(METADATA,
                                 result_dir = DATA_DIR)

In [ ]:
#This is the magic step, it loads the data to R and applies a function at the same time, in this case, the asleep annotation.
dt <- load_ethoscope(query,
                     reference_hour = 9.0, 
                     FUN = sleep_annotation,
                     cache = CACHE)

In [ ]:
#to include baseline days if there are multiple conditions 
dt[,t:=t+days(xmv(baseline_days))]

In [ ]:
dt<-dt[xmv(status)=="OK" ]
dt<-dt[xmv(sex)=="F" ]

In [ ]:
dt_curated <- curate_dead_animals(dt)
summary(dt_curated)

In [ ]:
#to add day number, and light phase
dt_curated [,day:=floor(t/days(1))]
dt_curated [,phase:=ifelse(t %% hours(24)>hours(12),"Dark","Light")]
dt_curated [,phase:=factor(phase, levels= c("Light","Dark"))]
gc()

In [ ]:
#time trimming to include only releavant
dt_curated <- dt[t >days(7) & t< days(10)]

In [ ]:
#selecting only for the time when rebound would occur - first 3 hours following SD
dt_rebound <- dt_curated[t >days(9) & t< days(9.125)]

In [ ]:
summary_dt <- dt_rebound[, .(sleep_fraction = mean(sleeping)), by=id]
summary_dt <- rejoin(summary_dt)

In [ ]:
ggplot(summary_dt, aes(x=as.factor(sdi), y=sleep_fraction, colour = as.factor(sdi))) + 
        geom_boxplot(outlier.colour = NA)+
        geom_jitter(alpha=.5) +
        #facet_grid(mating_status ~ .) +
        scale_y_continuous(name= "Sleep",labels = scales::percent, limits = c(-0.1,1)) +
        scale_x_discrete(name= "", labels = NULL) +
        theme_bw() +
        theme(legend.position = "bottom") +
        geom_jitter(height = 0) +
        stat_n_text() +
        labs(title="Female CrzR mut x CS treatment night-time", y="sleep") +
        theme_bw() +
        scale_color_hue() +
        scale_fill_hue() +
        stat_summary(fun.data=mean_cl_boot, geom="errorbar",size=2)+
        stat_summary(fun.data=mean_cl_boot, geom="point", size=4, shape=1) 